# Notebook 01 — Data Loading & QA Dataset (Kaggle T4)

**Dataset**: Chest X-Ray Pneumonia (`paultimothymooney/chest-xray-pneumonia`)

## Required Inputs (Kaggle → + Add Input)
1. `paultimothymooney/chest-xray-pneumonia` — the images

## Required Secrets
- `GROQ_API_KEY` — from https://console.groq.com (free tier)
- `HF_TOKEN` — HuggingFace token with MedGemma access

In [11]:
!pip install -q groq tqdm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.5 MB/s eta 0:00:00a 0:00:01


In [1]:
import os; print(os.listdir('/kaggle/input'))

['datasets']


In [ ]:
import os, sys, subprocess, glob

WORKING_DIR = '/kaggle/working'

# Locate the Chest X-Ray Pneumonia dataset root (contains train/val/test)
# Kaggle mounts it at /kaggle/input/chest-xray-pneumonia/chest_xray
search_dirs = glob.glob('/kaggle/input/**/train/NORMAL', recursive=True)

if not search_dirs:
    print('Contents of /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            print(f'  DIR: {os.path.join(root, d)}')
        for f in files[:5]:
            print(f'  FILE: {os.path.join(root, f)}')
        if root.count(os.sep) > 6:
            break
    raise RuntimeError(
        'Pneumonia dataset not found.\n'
        'Add it via + Add Input → paultimothymooney/chest-xray-pneumonia'
    )

# The dataset root is two levels above train/NORMAL
DATASET_ROOT = os.path.dirname(os.path.dirname(search_dirs[0]))
print(f'✓ Dataset root: {DATASET_ROOT}')

for split in ('train', 'val', 'test'):
    split_dir = os.path.join(DATASET_ROOT, split)
    if os.path.isdir(split_dir):
        n_normal    = len(glob.glob(os.path.join(split_dir, 'NORMAL', '*')))
        n_pneumonia = len(glob.glob(os.path.join(split_dir, 'PNEUMONIA', '*')))
        print(f'  {split:5s}: {n_normal} NORMAL  |  {n_pneumonia} PNEUMONIA')

In [5]:
# Get API keys from Kaggle secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret('GROQ_API_KEY')
HF_TOKEN = user_secrets.get_secret('HF_TOKEN')

print('✓ Secrets loaded')

✓ Secrets loaded


In [ ]:
# Clone repo
REPO_PATH = os.path.join(WORKING_DIR, 'cxr-rag-system')
if not os.path.exists(REPO_PATH):
    print('Cloning repository...')
    subprocess.run(
        ['git', 'clone', '-q', 'https://github.com/BASEL213/cxr-rag-system.git', REPO_PATH],
        check=True,
    )
else:
    print('Updating repository...')
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)
print('✓ Repository ready')

In [ ]:
# Verify images exist
sample_images = (
    glob.glob(os.path.join(DATASET_ROOT, '**', '*.jpeg'), recursive=True)[:3] +
    glob.glob(os.path.join(DATASET_ROOT, '**', '*.jpg'), recursive=True)[:3] +
    glob.glob(os.path.join(DATASET_ROOT, '**', '*.png'), recursive=True)[:3]
)
if not sample_images:
    raise RuntimeError(f'No images found under {DATASET_ROOT}')
print(f'✓ Sample image: {sample_images[0]}')

In [ ]:
# Load dataset using PneumoniaLoader
from src.data.pneumonia_loader import PneumoniaLoader

loader = PneumoniaLoader(images_dir=DATASET_ROOT)
df = loader.load()

print(f'Total images loaded: {len(df)}')
print(f'\nBy split:\n{df.groupby("split").size()}')
print(f'\nBy label:\n{df.groupby("label").size()}')
print(f'\nBy subtype:\n{df.groupby("subtype").size()}')
print(f'\nSample rows:')
print(df[['study_id', 'label', 'subtype', 'split', 'image_path']].head(5))

In [ ]:
import pandas as pd

# Use the dataset's native train/val/test splits
train_df, val_df, test_df = loader.train_val_test()

# Save corpus CSV (all splits, used by retriever notebooks)
corpus_path = os.path.join(WORKING_DIR, 'reports_corpus.csv')
df.to_csv(corpus_path, index=False)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Saved corpus → {corpus_path}')

In [ ]:
# QA Dataset Generation via Groq
# Uses synthetic impressions derived from class labels (NORMAL / PNEUMONIA_BACTERIAL / PNEUMONIA_VIRAL)
from src.data.qa_creator import QACreator

creator = QACreator(groq_api_key=GROQ_API_KEY)
QA_OUTPUT = os.path.join(WORKING_DIR, 'qa_dataset.jsonl')

# Limit to first 200 studies (Groq free-tier rate limit)
# The QA creator uses impression+findings to select clinically relevant questions
pairs = creator.generate_dataset(
    df=train_df,
    output_path=QA_OUTPUT,
    max_studies=200,
)

print(f'Generated {len(pairs)} QA pairs → {QA_OUTPUT}')

In [ ]:
# Stats and samples
import json

qa_df = pd.read_json(QA_OUTPUT, lines=True)
print(f'Total pairs   : {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'\nBy category:\n{qa_df["category"].value_counts()}')

print('\n=== Sample QA Pairs ===')
for s in qa_df.head(3).to_dict('records'):
    print(f"Q: {s['question']}")
    print(f"A: {s['answer']}\n")

print(f'\n✓ All outputs saved in {WORKING_DIR}')